In [2]:
import pandas as pd
import numpy as np

### Load Data

In [6]:
products=pd.read_csv('products.csv')
customers=pd.read_csv('customers.csv')
campaigns=pd.read_csv('campaigns.csv')
sales=pd.read_csv('sales_transactions.csv')

In [7]:
products.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   product_id         100 non-null    object 
 1   product_name       100 non-null    object 
 2   category           100 non-null    object 
 3   standard_price     100 non-null    float64
 4   cost_price         100 non-null    float64
 5   margin_percentage  100 non-null    float64
 6   active_status      100 non-null    object 
dtypes: float64(3), object(4)
memory usage: 5.6+ KB


In [8]:
customers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   customer_id          2000 non-null   object
 1   customer_type        2000 non-null   object
 2   segment              2000 non-null   object
 3   acquisition_channel  2000 non-null   object
 4   customer_since       2000 non-null   object
 5   location_category    2000 non-null   object
 6   age_band             1388 non-null   object
 7   business_size        612 non-null    object
dtypes: object(8)
memory usage: 125.1+ KB


In [9]:
campaigns.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   campaign_id    20 non-null     object 
 1   campaign_name  20 non-null     object 
 2   start_date     20 non-null     object 
 3   end_date       20 non-null     object 
 4   offer_type     20 non-null     object 
 5   campaign_cost  20 non-null     float64
dtypes: float64(1), object(5)
memory usage: 1.1+ KB


In [10]:
sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   transaction_id       50000 non-null  object 
 1   transaction_date     50000 non-null  object 
 2   customer_id          50000 non-null  object 
 3   product_id           50000 non-null  object 
 4   branch_id            50000 non-null  object 
 5   channel              50000 non-null  object 
 6   quantity             50000 non-null  int64  
 7   unit_price           50000 non-null  float64
 8   discount_percentage  50000 non-null  float64
 9   gross_revenue        50000 non-null  float64
 10  net_revenue          50000 non-null  float64
 11  cost                 50000 non-null  float64
 12  gross_margin         50000 non-null  float64
 13  payment_status       50000 non-null  object 
 14  campaign_id          9803 non-null   object 
dtypes: float64(6), int64(1), object(8)
m

In [11]:
sales["transaction_date"] = pd.to_datetime(sales["transaction_date"])
customers["customer_since"] = pd.to_datetime(customers["customer_since"])
campaigns["start_date"] = pd.to_datetime(campaigns["start_date"])
campaigns["end_date"] = pd.to_datetime(campaigns["end_date"])

### Merge age_band / business_size into one field

In [12]:
customers["customer_attribute"] = customers["age_band"].fillna(customers["business_size"])
customers["attribute_type"] = np.where(
    customers["age_band"].notna(), "age_band",
    np.where(customers["business_size"].notna(), "business_size", "none")
)
customers = customers.drop(columns=["age_band", "business_size"])

### Handling Missing Values

In [13]:
customers.isna().sum()

customer_id            0
customer_type          0
segment                0
acquisition_channel    0
customer_since         0
location_category      0
customer_attribute     0
attribute_type         0
dtype: int64

In [14]:
products.isna().sum()

product_id           0
product_name         0
category             0
standard_price       0
cost_price           0
margin_percentage    0
active_status        0
dtype: int64

In [15]:
campaigns.isna().sum()

campaign_id      0
campaign_name    0
start_date       0
end_date         0
offer_type       0
campaign_cost    0
dtype: int64

In [16]:
sales.isna().sum()

transaction_id             0
transaction_date           0
customer_id                0
product_id                 0
branch_id                  0
channel                    0
quantity                   0
unit_price                 0
discount_percentage        0
gross_revenue              0
net_revenue                0
cost                       0
gross_margin               0
payment_status             0
campaign_id            40197
dtype: int64

In [17]:
sales["has_campaign"] = sales["campaign_id"].notna()
sales["campaign_id"] = sales["campaign_id"].fillna("No Campaign")

In [18]:
sales.head()

,transaction_id,transaction_date,customer_id,product_id,branch_id,channel,quantity,unit_price,discount_percentage,gross_revenue,net_revenue,cost,gross_margin,payment_status,campaign_id,has_campaign
0,TXN034365,2024-01-01,C00155,P0099,BR06,Online,6,251.81,5.0,1510.86,1435.32,950.22,485.10,Paid,No Campaign,False
1,TXN010962,2024-01-01,C00043,P0006,BR03,Branch,1,47.10,10.0,47.10,42.39,25.71,16.68,Paid,No Campaign,False
2,TXN007598,2024-01-01,C01252,P0004,BR05,Dealer,1,82.84,20.0,82.84,66.27,55.08,11.19,Paid,No Campaign,False
3,TXN038126,2024-01-01,C01733,P0096,BR02,Branch,1,35.47,10.0,35.47,31.92,25.23,6.69,Paid,No Campaign,False
4,TXN018144,2024-01-01,C01923,P0049,BR03,Direct Sales,3,141.88,20.0,425.64,340.51,399.60,-59.09,Paid,No Campaign,False


### JOIN: sales + products + customers + campaigns

In [28]:
master = sales.merge(
    products[["product_id", "product_name", "category","standard_price","cost_price","margin_percentage", "active_status"]],
    on="product_id", how="left"
)
master = master.merge(
    customers[["customer_id", "customer_type", "segment", "acquisition_channel",
               "customer_since", "location_category", "customer_attribute", "attribute_type"]],
    on="customer_id", how="left"
)
master = master.merge(
    campaigns[["campaign_id", "campaign_name","start_date","end_date", "offer_type","campaign_cost"]],
    on="campaign_id", how="left"
)
master["campaign_name"] = master["campaign_name"].fillna("No Campaign")
master["offer_type"] = master["offer_type"].fillna("No")
master = master.rename(columns={"start_date": "campaign_start_date", "end_date": "campaign_end_date"})

### Derives Columns

In [29]:
master["transaction_year"] = master["transaction_date"].dt.year
master["transaction_month"] = master["transaction_date"].dt.month
master["transaction_year_month"] = master["transaction_date"].dt.to_period("M").astype(str)

master["discount_band"] = pd.cut(
    master["discount_percentage"],
    bins=[-0.01, 0, 5, 10, 15, 20, 100],
    labels=["0%", "1-5%", "6-10%", "11-15%", "16-20%", "20%+"]
)

first_purchase = master.groupby("customer_id")["transaction_date"].transform("min")
master["customer_since"] = np.where(
    master["customer_since"] > first_purchase,
    first_purchase,
    master["customer_since"]
)
master["customer_since"] = pd.to_datetime(master["customer_since"])

master["customer_tenure_days"] = (master["transaction_date"] - master["customer_since"]).dt.days

master["margin_percentage_tx"] = np.where(
    master["net_revenue"] > 0,
    (master["gross_margin"] / master["net_revenue"] * 100).round(2),
    0
)

### Validation Check

In [30]:
print("=" * 60)
print("VALIDATION REPORT")
print("=" * 60)
print(f"Total rows in master_sales: {len(master):,}")
print(f"Date range: {master['transaction_date'].min().date()} to {master['transaction_date'].max().date()}")
print()

print("-- Referential integrity --")
print("Unmatched product_id:", master["product_name"].isna().sum())
print("Unmatched customer_id:", master["customer_type"].isna().sum())
print()

print("-- Negative value checks --")
print("Negative net_revenue:", (master["net_revenue"] < 0).sum())
print("Negative gross_margin:", (master["gross_margin"] < 0).sum(), "(expected: some, due to heavy discounting)")
print("Negative customer_tenure_days:", (master["customer_tenure_days"] < 0).sum(),
      "(should be 0 - a sale can't happen before the customer joined)")
print()

print("-- Missing values per column --")
null_counts = master.isnull().sum()
print(null_counts[null_counts > 0] if null_counts.sum() > 0 else "None")
print()

print("-- Category summary --")
print("payment_status:\n", master["payment_status"].value_counts())
print()
print("has_campaign:\n", master["has_campaign"].value_counts())

VALIDATION REPORT
Total rows in master_sales: 50,000
Date range: 2024-01-01 to 2025-12-31

-- Referential integrity --
Unmatched product_id: 0
Unmatched customer_id: 0

-- Negative value checks --
Negative net_revenue: 0
Negative gross_margin: 2230 (expected: some, due to heavy discounting)
Negative customer_tenure_days: 0 (should be 0 - a sale can't happen before the customer joined)

-- Missing values per column --
campaign_start_date    40197
campaign_end_date      40197
campaign_cost          40197
dtype: int64

-- Category summary --
payment_status:
 payment_status
Paid              41930
Overdue            3630
Pending            2967
Partially Paid     1473
Name: count, dtype: int64

has_campaign:
 has_campaign
False    40197
True      9803
Name: count, dtype: int64


### Save

In [41]:
import os
RAW_DIR = "C:/Users/SHINI/revenue_analytics_project/data/raw"
OUT_DIR = "C:/Users/SHINI/revenue_analytics_project/data"

os.makedirs(f"{OUT_DIR}/processed", exist_ok=True)
master.to_csv(f"{OUT_DIR}/processed/master_sales.csv", index=False)

print()
print(f"Saved master_sales.csv -> {len(master):,} rows, {len(master.columns)} columns")
print("Columns:", list(master.columns))

corrected_since = master.groupby("customer_id")["customer_since"].min().reset_index()
customers_clean = customers.drop(columns=["customer_since"]).merge(corrected_since, on="customer_id", how="left")

no_tx_mask = customers_clean["customer_since"].isna()
if no_tx_mask.any():
    fallback = customers.set_index("customer_id").loc[customers_clean.loc[no_tx_mask, "customer_id"], "customer_since"]
    customers_clean.loc[no_tx_mask, "customer_since"] = fallback.values

customers_clean.to_csv(f"{OUT_DIR}/customers_cleaned.csv", index=False)
print(f"Saved customers_cleaned.csv -> {len(customers_clean):,} rows (customer_since corrected)")

print()
print("-- Post-fix check --")
print("Negative customer_tenure_days now:", (master["customer_tenure_days"] < 0).sum())

print()
print("Absolute path used:", os.path.abspath(f"{OUT_DIR}/processed/master_sales.csv"))
print("File exists?", os.path.exists(f"{OUT_DIR}/processed/master_sales.csv"))
print()
print("Contents of OUT_DIR/processed:")
print(os.listdir(f"{OUT_DIR}/processed"))


Saved master_sales.csv -> 50,000 rows, 40 columns
Columns: ['transaction_id', 'transaction_date', 'customer_id', 'product_id', 'branch_id', 'channel', 'quantity', 'unit_price', 'discount_percentage', 'gross_revenue', 'net_revenue', 'cost', 'gross_margin', 'payment_status', 'campaign_id', 'has_campaign', 'product_name', 'category', 'standard_price', 'cost_price', 'margin_percentage', 'active_status', 'customer_type', 'segment', 'acquisition_channel', 'customer_since', 'location_category', 'customer_attribute', 'attribute_type', 'campaign_name', 'campaign_start_date', 'campaign_end_date', 'offer_type', 'campaign_cost', 'transaction_year', 'transaction_month', 'transaction_year_month', 'discount_band', 'customer_tenure_days', 'margin_percentage_tx']
Saved customers_cleaned.csv -> 2,000 rows (customer_since corrected)

-- Post-fix check --
Negative customer_tenure_days now: 0

Absolute path used: C:\Users\SHINI\revenue_analytics_project\data\processed\master_sales.csv
File exists? True

C

In [42]:
print("Absolute path used:", os.path.abspath(f"{OUT_DIR}/processed/master_sales.csv"))
print("File exists?", os.path.exists(f"{OUT_DIR}/processed/master_sales.csv"))
print()
print("Contents of OUT_DIR/processed:")
print(os.listdir(f"{OUT_DIR}/processed"))

Absolute path used: C:\Users\SHINI\revenue_analytics_project\data\processed\master_sales.csv
File exists? True

Contents of OUT_DIR/processed:
['.~lock.master_sales.csv#', 'master_sales.csv']


In [43]:
master = pd.read_csv("C:/Users/SHINI/revenue_analytics_project/data/processed/master_sales.csv")

C:\Users\SHINI\AppData\Local\Temp\ipykernel_13516\3669382234.py:1: DtypeWarning: Columns (30,31) have mixed types. Specify dtype option on import or set low_memory=False.
  master = pd.read_csv("C:/Users/SHINI/revenue_analytics_project/data/processed/master_sales.csv")


In [44]:
print(master.columns.tolist())

['transaction_id', 'transaction_date', 'customer_id', 'product_id', 'branch_id', 'channel', 'quantity', 'unit_price', 'discount_percentage', 'gross_revenue', 'net_revenue', 'cost', 'gross_margin', 'payment_status', 'campaign_id', 'has_campaign', 'product_name', 'category', 'standard_price', 'cost_price', 'margin_percentage', 'active_status', 'customer_type', 'segment', 'acquisition_channel', 'customer_since', 'location_category', 'customer_attribute', 'attribute_type', 'campaign_name', 'campaign_start_date', 'campaign_end_date', 'offer_type', 'campaign_cost', 'transaction_year', 'transaction_month', 'transaction_year_month', 'discount_band', 'customer_tenure_days', 'margin_percentage_tx']


In [45]:
master.head()

,transaction_id,transaction_date,customer_id,product_id,branch_id,channel,quantity,unit_price,discount_percentage,gross_revenue,...,campaign_start_date,campaign_end_date,offer_type,campaign_cost,transaction_year,transaction_month,transaction_year_month,discount_band,customer_tenure_days,margin_percentage_tx
0,TXN034365,2024-01-01,C00155,P0099,BR06,Online,6,251.81,5.0,1510.86,...,NaN,NaN,No offer,NaN,2024,1,2024-01,1-5%,0,33.80
1,TXN010962,2024-01-01,C00043,P0006,BR03,Branch,1,47.10,10.0,47.10,...,NaN,NaN,No offer,NaN,2024,1,2024-01,6-10%,0,39.35
2,TXN007598,2024-01-01,C01252,P0004,BR05,Dealer,1,82.84,20.0,82.84,...,NaN,NaN,No offer,NaN,2024,1,2024-01,16-20%,0,16.89
3,TXN038126,2024-01-01,C01733,P0096,BR02,Branch,1,35.47,10.0,35.47,...,NaN,NaN,No offer,NaN,2024,1,2024-01,6-10%,652,20.96
4,TXN018144,2024-01-01,C01923,P0049,BR03,Direct Sales,3,141.88,20.0,425.64,...,NaN,NaN,No offer,NaN,2024,1,2024-01,16-20%,0,-17.35


In [46]:
master.isna().sum()

transaction_id                0
transaction_date              0
customer_id                   0
product_id                    0
branch_id                     0
channel                       0
quantity                      0
unit_price                    0
discount_percentage           0
gross_revenue                 0
net_revenue                   0
cost                          0
gross_margin                  0
payment_status                0
campaign_id                   0
has_campaign                  0
product_name                  0
category                      0
standard_price                0
cost_price                    0
margin_percentage             0
active_status                 0
customer_type                 0
segment                       0
acquisition_channel           0
customer_since                0
location_category             0
customer_attribute            0
attribute_type                0
campaign_name                 0
campaign_start_date       40197
campaign

In [37]:
master.shape

(50000, 40)

### Module 6: Revenue Leakage and Underperformance Analysis

In [6]:
df = pd.read_csv("master_sales.csv", low_memory=False)
df["transaction_date"] = pd.to_datetime(df["transaction_date"])
snapshot_date = df["transaction_date"].max()

print("="*70)
print("1. OVERDUE / AT-RISK REVENUE (Aging Analysis)")
print("="*70)

at_risk_statuses = ["Overdue", "Pending", "Partially Paid"]
at_risk = df[df["payment_status"].isin(at_risk_statuses)]

total_net_revenue = df["net_revenue"].sum()
at_risk_revenue = at_risk["net_revenue"].sum()

print(f"Total net revenue: {total_net_revenue:,.2f}")
print(f"At-risk revenue (Overdue+Pending+Partially Paid): {at_risk_revenue:,.2f} "
      f"({at_risk_revenue/total_net_revenue*100:.2f}% of total)")
print()
print("Breakdown by status:")
status_summary = df.groupby("payment_status")["net_revenue"].agg(["sum", "count"]).round(2)
status_summary["% of total revenue"] = (status_summary["sum"] / total_net_revenue * 100).round(2)
print(status_summary)

at_risk = at_risk.copy()
at_risk["days_outstanding"] = (snapshot_date - at_risk["transaction_date"]).dt.days
bins = [-1, 30, 60, 90, 10_000]
labels = ["0-30 days", "31-60 days", "61-90 days", "90+ days"]
at_risk["aging_bucket"] = pd.cut(at_risk["days_outstanding"], bins=bins, labels=labels)
print()
print("Aging buckets (at-risk transactions only):")
aging_summary = at_risk.groupby("aging_bucket", observed=True)["net_revenue"].agg(["sum", "count"]).round(2)
print(aging_summary)

1. OVERDUE / AT-RISK REVENUE (Aging Analysis)
Total net revenue: 18,113,801.64
At-risk revenue (Overdue+Pending+Partially Paid): 2,980,373.07 (16.45% of total)

Breakdown by status:
                        sum  count  % of total revenue
payment_status                                        
Overdue          1348108.73   3630                7.44
Paid            15133428.57  41930               83.55
Partially Paid    537440.50   1473                2.97
Pending          1094823.84   2967                6.04

Aging buckets (at-risk transactions only):
                     sum  count
aging_bucket                   
0-30 days      181156.17    480
31-60 days     162121.43    418
61-90 days     126870.44    364
90+ days      2510225.03   6808


In [7]:
print()
print("="*70)
print("2. EXCESSIVE DISCOUNTING (Threshold-Based Flags)")
print("="*70)

DISCOUNT_THRESHOLD = 20  
high_discount = df[df["discount_percentage"] > DISCOUNT_THRESHOLD]
low_discount = df[df["discount_percentage"] <= DISCOUNT_THRESHOLD]

print(f"Transactions with discount > {DISCOUNT_THRESHOLD}%: {len(high_discount):,} "
      f"({len(high_discount)/len(df)*100:.1f}% of all transactions)")
print(f"Revenue tied up in high-discount transactions: {high_discount['net_revenue'].sum():,.2f}")
print()
print(f"Avg margin % - high discount group: {high_discount['margin_percentage_tx'].mean():.2f}%")
print(f"Avg margin % - low/no discount group: {low_discount['margin_percentage_tx'].mean():.2f}%")
print(f"Avg quantity - high discount group: {high_discount['quantity'].mean():.2f}")
print(f"Avg quantity - low/no discount group: {low_discount['quantity'].mean():.2f}")

print()
print("Products most exposed to excessive discounting (by revenue lost to high discounts):")
prod_discount = high_discount.groupby("product_name")["net_revenue"].sum().sort_values(ascending=False).head(10)
print(prod_discount.round(2))


2. EXCESSIVE DISCOUNTING (Threshold-Based Flags)
Transactions with discount > 20%: 5,423 (10.8% of all transactions)
Revenue tied up in high-discount transactions: 1,614,675.05

Avg margin % - high discount group: 11.79%
Avg margin % - low/no discount group: 30.76%
Avg quantity - high discount group: 3.20
Avg quantity - low/no discount group: 3.19

Products most exposed to excessive discounting (by revenue lost to high discounts):
product_name
Pro Cutlery Set            138540.59
Everyday Denim Jacket       91915.46
Essential Tablet Sleeve     84001.90
Lite Webcam                 75592.03
Pro Webcam                  75314.74
Signature Lip Balm Set      63303.68
Advanced Cutlery Set        59528.29
Max Home Router             56093.06
Compact Football            54698.76
Deluxe Water Bottle         50170.61
Name: net_revenue, dtype: float64


In [8]:
print()
print("="*70)
print("3. LOW-MARGIN SALES")
print("="*70)

MARGIN_THRESHOLD = 10  
low_margin = df[df["margin_percentage_tx"] < MARGIN_THRESHOLD]

print(f"Transactions with margin % < {MARGIN_THRESHOLD}%: {len(low_margin):,} "
      f"({len(low_margin)/len(df)*100:.1f}% of all transactions)")
print(f"Revenue tied up in low-margin sales: {low_margin['net_revenue'].sum():,.2f} "
      f"({low_margin['net_revenue'].sum()/total_net_revenue*100:.2f}% of total revenue)")
print()
print("Low-margin revenue by category:")
low_margin_by_cat = low_margin.groupby("category")["net_revenue"].sum().sort_values(ascending=False)
print(low_margin_by_cat.round(2))
print()
print("Low-margin revenue by channel:")
low_margin_by_channel = low_margin.groupby("channel")["net_revenue"].sum().sort_values(ascending=False)
print(low_margin_by_channel.round(2))


3. LOW-MARGIN SALES
Transactions with margin % < 10%: 4,796 (9.6% of all transactions)
Revenue tied up in low-margin sales: 1,308,073.00 (7.22% of total revenue)

Low-margin revenue by category:
category
Sports & Outdoors         634121.87
Home & Living             219085.69
Electronics               204304.68
Apparel                    98142.86
Beauty & Personal Care     78663.81
Food & Beverage            73754.09
Name: net_revenue, dtype: float64

Low-margin revenue by channel:
channel
Online          448637.31
Branch          349938.27
Dealer          207403.44
Direct Sales    173059.57
Partner         129034.41
Name: net_revenue, dtype: float64


In [10]:
print()
print("="*70)
print("4. INACTIVE CUSTOMERS")
print("="*70)

INACTIVITY_DAYS = 90  
last_purchase = df.groupby("customer_id")["transaction_date"].max().reset_index()
last_purchase["days_since_last_purchase"] = (snapshot_date - last_purchase["transaction_date"]).dt.days
last_purchase["status"] = np.where(last_purchase["days_since_last_purchase"] > INACTIVITY_DAYS,
                                    "Inactive", "Active")

n_inactive = (last_purchase["status"] == "Inactive").sum()
n_customers = last_purchase.shape[0]
print(f"Inactivity threshold: no purchase in {INACTIVITY_DAYS}+ days")
print(f"Inactive customers: {n_inactive:,} out of {n_customers:,} "
      f"({n_inactive/n_customers*100:.2f}%)")

inactive_ids = last_purchase.loc[last_purchase["status"] == "Inactive", "customer_id"]
inactive_value = df[df["customer_id"].isin(inactive_ids)]["net_revenue"].sum()
print(f"Historical revenue generated by now-inactive customers: {inactive_value:,.2f}")


4. INACTIVE CUSTOMERS
Inactivity threshold: no purchase in 90+ days
Inactive customers: 148 out of 2,000 (7.40%)
Historical revenue generated by now-inactive customers: 901,341.10


In [11]:
print()
print("="*70)
print("5. WEAK BRANCHES / CHANNELS")
print("="*70)

branch_perf = df.groupby("branch_id").agg(
    total_revenue=("net_revenue", "sum"),
    avg_margin_pct=("margin_percentage_tx", "mean"),
    n_transactions=("transaction_id", "count")
).round(2)
branch_perf["revenue_vs_avg"] = (branch_perf["total_revenue"] - branch_perf["total_revenue"].mean()).round(2)
print("Branch performance:")
print(branch_perf.sort_values("total_revenue"))

weak_branches = branch_perf[
    (branch_perf["total_revenue"] < branch_perf["total_revenue"].mean()) &
    (branch_perf["avg_margin_pct"] < branch_perf["avg_margin_pct"].mean())
]
print()
print(f"Branches below average on BOTH revenue and margin (flagged as weak): {list(weak_branches.index)}")

print()
channel_perf = df.groupby("channel").agg(
    total_revenue=("net_revenue", "sum"),
    avg_margin_pct=("margin_percentage_tx", "mean"),
    n_transactions=("transaction_id", "count")
).round(2)
print("Channel performance:")
print(channel_perf.sort_values("total_revenue"))

weak_channels = channel_perf[
    (channel_perf["total_revenue"] < channel_perf["total_revenue"].mean()) &
    (channel_perf["avg_margin_pct"] < channel_perf["avg_margin_pct"].mean())
]
print(f"\nChannels below average on BOTH revenue and margin (flagged as weak): {list(weak_channels.index)}")


5. WEAK BRANCHES / CHANNELS
Branch performance:
           total_revenue  avg_margin_pct  n_transactions  revenue_vs_avg
branch_id                                                               
BR08          1401291.57           28.93            3986      -862933.64
BR07          1826339.55           28.95            4929      -437885.66
BR06          1847936.41           28.33            5139      -416288.80
BR05          2165589.80           28.69            6047       -98635.41
BR04          2417565.75           28.58            6374       153340.54
BR03          2540622.25           28.63            7053       276397.04
BR02          2664098.66           28.57            7522       399873.46
BR01          3250357.65           28.93            8950       986132.44

Branches below average on BOTH revenue and margin (flagged as weak): ['BR05', 'BR06']

Channel performance:
              total_revenue  avg_margin_pct  n_transactions
channel                                             

In [12]:
print()
print("="*70)
print("6. PRODUCT UNDERPERFORMANCE")
print("="*70)

product_perf = df.groupby("product_name").agg(
    total_revenue=("net_revenue", "sum"),
    total_margin=("gross_margin", "sum"),
    avg_margin_pct=("margin_percentage_tx", "mean"),
    n_transactions=("transaction_id", "count")
).round(2).sort_values("total_revenue", ascending=False)

product_perf["cumulative_revenue_pct"] = (product_perf["total_revenue"].cumsum() /
                                           product_perf["total_revenue"].sum() * 100).round(2)

underperformers = product_perf[product_perf["cumulative_revenue_pct"] > 80]
print(f"Total products: {len(product_perf)}")
print(f"Underperforming products (bottom ~20% of cumulative revenue - Pareto tail): {len(underperformers)}")
print(f"Combined revenue of underperformers: {underperformers['total_revenue'].sum():,.2f} "
      f"({underperformers['total_revenue'].sum()/total_net_revenue*100:.2f}% of total)")
print()
print("Sample of underperforming products:")
print(underperformers.head(10))

at_risk.to_csv("leakage_overdue_transactions.csv", index=False)
high_discount.to_csv("leakage_excessive_discounts.csv", index=False)
low_margin.to_csv("leakage_low_margin_sales.csv", index=False)
last_purchase.to_csv("leakage_customer_activity_status.csv", index=False)
branch_perf.to_csv("leakage_branch_performance.csv")
channel_perf.to_csv("leakage_channel_performance.csv")
product_perf.to_csv("leakage_product_performance.csv")

print()
print("All flagged datasets saved as CSV files for use in the app / report appendix.")


6. PRODUCT UNDERPERFORMANCE
Total products: 85
Underperforming products (bottom ~20% of cumulative revenue - Pareto tail): 58
Combined revenue of underperformers: 3,729,961.59 (20.59% of total)

Sample of underperforming products:
                            total_revenue  total_margin  avg_margin_pct  \
product_name                                                              
Classic Jump Rope               213593.85      39319.15           17.89   
Everyday Yoga Mat               199142.64      59893.53           29.27   
Advanced Water Bottle           179953.35      72682.95           39.72   
Essential USB-C Hub             166049.23      26814.53           15.21   
Deluxe Laptop Stand             161805.10      74545.63           45.44   
Prime Sparkling Water Pack      135267.01      31230.91           22.26   
Compact Rain Jacket             129905.21      34948.04           26.25   
Ultra Camping Tent              124862.33       2811.00            1.38   
Ultra Cushion Cove

In [19]:
total_net_revenue = df["net_revenue"].sum()

print("REVENUE LEAKAGE & UNDERPERFORMANCE - SUMMARY RESULTS")
print("="*60)

at_risk = df[df["payment_status"].isin(["Overdue", "Pending", "Partially Paid"])]
at_risk_revenue = at_risk["net_revenue"].sum()
old_at_risk = at_risk[(snapshot_date - at_risk["transaction_date"]).dt.days > 90]["net_revenue"].sum()

print(f"\n1. Overdue/At-Risk Revenue")
print(f"   Total net revenue: Rs. {total_net_revenue:,.2f}")
print(f"   At-risk revenue: Rs. {at_risk_revenue:,.2f} ({at_risk_revenue/total_net_revenue*100:.2f}% of total)")
print(f"   Of which, 90+ days old: Rs. {old_at_risk:,.2f} ({old_at_risk/at_risk_revenue*100:.1f}% of at-risk)")

high_disc = df[df["discount_percentage"] > 20]
low_disc = df[df["discount_percentage"] <= 20]
print(f"\n2. Excessive Discounting (>20%)")
print(f"   Transactions: {len(high_disc):,} ({len(high_disc)/len(df)*100:.1f}% of all)")
print(f"   Avg margin - high discount: {high_disc['margin_percentage_tx'].mean():.2f}%  vs  low discount: {low_disc['margin_percentage_tx'].mean():.2f}%")
print(f"   Avg quantity - high discount: {high_disc['quantity'].mean():.2f}  vs  low discount: {low_disc['quantity'].mean():.2f}")

low_margin = df[df["margin_percentage_tx"] < 10]
print(f"\n3. Low-Margin Sales (<10%)")
print(f"   Transactions: {len(low_margin):,} ({len(low_margin)/len(df)*100:.1f}% of all)")
print(f"   Revenue tied up: Rs. {low_margin['net_revenue'].sum():,.2f} ({low_margin['net_revenue'].sum()/total_net_revenue*100:.2f}% of total)")
print(f"   Top category: {low_margin.groupby('category')['net_revenue'].sum().idxmax()}")
print(f"   Top channel: {low_margin.groupby('channel')['net_revenue'].sum().idxmax()}")

last_purchase = df.groupby("customer_id")["transaction_date"].max()
days_since = (snapshot_date - last_purchase).dt.days
inactive_ids = days_since[days_since > 90].index
n_customers = last_purchase.shape[0]
inactive_revenue = df[df["customer_id"].isin(inactive_ids)]["net_revenue"].sum()
print(f"\n4. Inactive Customers (90+ days)")
print(f"   Inactive: {len(inactive_ids):,} of {n_customers:,} ({len(inactive_ids)/n_customers*100:.2f}%)")
print(f"   Historical revenue from inactive customers: Rs. {inactive_revenue:,.2f}")

branch_perf = df.groupby("branch_id").agg(revenue=("net_revenue","sum"), margin=("margin_percentage_tx","mean"))
weak_branches = branch_perf[(branch_perf["revenue"] < branch_perf["revenue"].mean()) &
                             (branch_perf["margin"] < branch_perf["margin"].mean())]
channel_perf = df.groupby("channel").agg(revenue=("net_revenue","sum"), margin=("margin_percentage_tx","mean"))
weak_channels = channel_perf[(channel_perf["revenue"] < channel_perf["revenue"].mean()) &
                              (channel_perf["margin"] < channel_perf["margin"].mean())]
print(f"\n5. Weak Branches/Channels")
print(f"   Weak branches (below avg revenue AND margin): {list(weak_branches.index)}")
print(f"   Weak channels (below avg revenue AND margin): {list(weak_channels.index)}")

product_perf = df.groupby("product_name")["net_revenue"].sum().sort_values(ascending=False)
cum_pct = (product_perf.cumsum() / product_perf.sum() * 100)
underperformers = cum_pct[cum_pct > 80]
print(f"\n6. Product Underperformance")
print(f"   Total products: {product_perf.shape[0]}")
print(f"   Underperforming (bottom 20% cumulative revenue): {len(underperformers)} ({len(underperformers)/product_perf.shape[0]*100:.0f}%)")
print(f"   Combined revenue of underperformers: Rs. {product_perf[underperformers.index].sum():,.2f} ({product_perf[underperformers.index].sum()/total_net_revenue*100:.2f}% of total)")

print("\n" + "="*60)
print("END OF SUMMARY")

REVENUE LEAKAGE & UNDERPERFORMANCE - SUMMARY RESULTS

1. Overdue/At-Risk Revenue
   Total net revenue: Rs. 18,113,801.64
   At-risk revenue: Rs. 2,980,373.07 (16.45% of total)
   Of which, 90+ days old: Rs. 2,510,225.03 (84.2% of at-risk)

2. Excessive Discounting (>20%)
   Transactions: 5,423 (10.8% of all)
   Avg margin - high discount: 11.79%  vs  low discount: 30.76%
   Avg quantity - high discount: 3.20  vs  low discount: 3.19

3. Low-Margin Sales (<10%)
   Transactions: 4,796 (9.6% of all)
   Revenue tied up: Rs. 1,308,073.00 (7.22% of total)
   Top category: Sports & Outdoors
   Top channel: Online

4. Inactive Customers (90+ days)
   Inactive: 148 of 2,000 (7.40%)
   Historical revenue from inactive customers: Rs. 901,341.10

5. Weak Branches/Channels
   Weak branches (below avg revenue AND margin): ['BR05', 'BR06']
   Weak channels (below avg revenue AND margin): ['Dealer']

6. Product Underperformance
   Total products: 85
   Underperforming (bottom 20% cumulative revenue): 5